# Day 5 — Statistics Fundamentals: Descriptive & Inferential Statistics

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("titanic.csv")
df.columns = df.columns.str.strip().str.lower()
print("Dataset shape:", df.shape)
display(df.head())
print("Columns:", list(df.columns))


Dataset shape: (1309, 11)


,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,survived
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,1
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,1
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,0
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,0
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,0


Columns: ['pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'survived']


## 1. Descriptive Statistics

We calculate mean, median, standard deviation, variance, minimum and maximum for important numerical variables.


In [2]:
cols = ["age","fare","sibsp","parch","pclass","survived"]
cols = [c for c in cols if c in df.columns]

print("Missing values:")
display(df[cols].isna().sum().to_frame("missing"))

stats_table = pd.DataFrame({
    "mean": df[cols].mean(),
    "median": df[cols].median(),
    "std": df[cols].std(),
    "variance": df[cols].var(),
    "min": df[cols].min(),
    "max": df[cols].max()
})
display(stats_table.round(3))


Missing values:


,missing
age,263
fare,1
sibsp,0
parch,0
pclass,0
survived,0


,mean,median,std,variance,min,max
age,29.881,28.000,14.413,207.749,0.167,80.000
fare,33.295,14.454,51.759,2678.960,0.000,512.329
sibsp,0.499,0.000,1.042,1.085,0.000,8.000
parch,0.385,0.000,0.866,0.749,0.000,9.000
pclass,2.295,3.000,0.838,0.702,1.000,3.000
survived,0.382,0.000,0.486,0.236,0.000,1.000


## 2. Overall Survival Rate

In [3]:
survival_rate = df["survived"].mean()
print("Total passengers:", df["survived"].count())
print("Survivors:", int(df["survived"].sum()))
print(f"Overall survival rate: {survival_rate*100:.2f}%")


Total passengers: 1309
Survivors: 500
Overall survival rate: 38.20%


## 3. Descriptive Statistics by Passenger Class

In [4]:
class_stats = (df.groupby("pclass")
    .agg(passengers=("survived","count"),
         survival_rate=("survived","mean"),
         mean_fare=("fare","mean"),
         median_fare=("fare","median"),
         mean_age=("age","mean"),
         median_age=("age","median"))
    .reset_index())
class_stats["survival_rate"] *= 100
display(class_stats.round(2))


,pclass,passengers,survival_rate,mean_fare,median_fare,mean_age,median_age
0,1,323,61.92,87.51,60.00,39.16,39.0
1,2,277,42.96,21.18,15.05,29.51,29.0
2,3,709,25.53,13.30,8.05,24.82,24.0


## 4. Hypothesis Test — First-Class vs Third-Class Mean Fare

**Research question:** Do first-class and third-class passengers have different average fares?

**H₀:** The mean fare is the same for first and third class.

**H₁:** The mean fare is different between first and third class.

We use Welch's independent two-sample t-test because it does not assume equal variances.

**Significance level:** α = 0.05


In [5]:
first = df.loc[df["pclass"] == 1, "fare"].dropna()
third = df.loc[df["pclass"] == 3, "fare"].dropna()

t_stat, p_value = stats.ttest_ind(first, third, equal_var=False)

print(f"First-class n = {len(first)}, mean = {first.mean():.2f}")
print(f"Third-class n = {len(third)}, mean = {third.mean():.2f}")
print(f"t-statistic = {t_stat:.4f}")
print(f"p-value = {p_value:.6g}")

if p_value < 0.05:
    print("Decision: Reject H0.")
    print("There is statistically significant evidence that first- and third-class mean fares differ.")
else:
    print("Decision: Fail to reject H0.")
    print("There is not enough evidence to conclude that the mean fares differ.")


First-class n = 323, mean = 87.51
Third-class n = 708, mean = 13.30
t-statistic = 16.5013
p-value = 5.82443e-45
Decision: Reject H0.
There is statistically significant evidence that first- and third-class mean fares differ.


## 5. 95% Confidence Interval

The interval below estimates the plausible range for:

**Mean fare (First Class) − Mean fare (Third Class)**


In [6]:
m1, m3 = first.mean(), third.mean()
v1, v3 = first.var(ddof=1), third.var(ddof=1)
n1, n3 = len(first), len(third)

difference = m1 - m3
se = np.sqrt(v1/n1 + v3/n3)
df_w = (v1/n1 + v3/n3)**2 / (((v1/n1)**2/(n1-1)) + ((v3/n3)**2/(n3-1)))
critical = stats.t.ppf(0.975, df_w)
margin = critical * se
ci_low, ci_high = difference-margin, difference+margin

print(f"Difference in means: {difference:.2f}")
print(f"95% CI: ({ci_low:.2f}, {ci_high:.2f})")
print(f"Welch degrees of freedom: {df_w:.2f}")


Difference in means: 74.21
95% CI: (65.36, 83.05)
Welch degrees of freedom: 328.01


## 6. Plain-Language Interpretation

The t-test evaluates whether the observed difference in average fares between first and third class is likely to be due to random sampling variation. A p-value below 0.05 leads to rejection of the null hypothesis.

The 95% confidence interval gives a plausible range for the population difference in mean fares.

**Limitation:** Titanic fares are strongly right-skewed and contain outliers, so the t-test assumptions are not perfect. Welch's test is useful for unequal variances, but a non-parametric test such as Mann–Whitney U could also be considered.


In [7]:
print("FINAL CONCLUSION")
if p_value < 0.05:
    print("Reject H0: first- and third-class mean fares are statistically different at the 5% level.")
else:
    print("Fail to reject H0: insufficient evidence of a difference at the 5% level.")
print(f"95% CI for mean-fare difference: ({ci_low:.2f}, {ci_high:.2f})")


FINAL CONCLUSION
Reject H0: first- and third-class mean fares are statistically different at the 5% level.
95% CI for mean-fare difference: (65.36, 83.05)


# Key Findings

- Descriptive statistics summarize age, fare, family-related variables, class and survival.
- Overall survival rate is calculated directly from the Titanic data.
- Passenger-class statistics compare survival, fare and age.
- Welch's t-test compares first- and third-class mean fares.
- A 95% confidence interval quantifies the estimated difference.
- Fare skewness and outliers are documented as a limitation.

# Interview Questions

### What does a p-value mean?
A p-value measures how compatible the observed data are with the null hypothesis. A small p-value indicates that the observed result would be relatively unlikely if the null hypothesis were true. It is not the probability that the null hypothesis is true.

### What is the difference between Type I and Type II errors?
A Type I error is rejecting a true null hypothesis (false positive). A Type II error is failing to reject a false null hypothesis (false negative).

### When would you use a t-test instead of a chi-square test?
Use a t-test to compare a numerical mean between groups, such as average fare between passenger classes. Use a chi-square test to examine relationships between categorical variables, such as gender and survival.
